In [7]:
# Imports
%pip install selenium webdriver-manager beautifulsoup4
%pip install undetected-chromedriver
%pip install pandas


from bs4 import BeautifulSoup
import csv
import re
import os
import time
import logging
import urllib.parse
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from itertools import islice


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [4]:
# Set up logging
logging.basicConfig(
    filename="critic-reviews-metacritic/scrape_log.txt",
    filemode="a",
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    level=logging.INFO,
)

In [40]:
movies = {
    'The Shawshank Redemption': 1994, 
    'Parasite': 2019,
    'Star Wars: Episode V - The Empire Strikes Back': 1980, 
    'The Godfather': 1972,
    'Top Gun: Maverick': 2022, 
    'The Godfather: Part II': 1974,
    'Spider-Man: Into the Spider-Verse': 2018, 
    'Raiders of the Lost Ark': 1981,
    'The Dark Knight': 2008, 
    'Mission: Impossible Dead Reckoning': 2023,
    '12 Angry Men': 1957, 
    'Pulp Fiction': 1994, 
    "Schindler's List": 1993,
    'Mad Max: Fury Road': 2015, 
    'Harakiri': 1962,
    'The Lord of the Rings: The Return of the King': 2003, 
    'Spirited Away': 2001,
    'Ikiru': 1952, 'Inside Out': 2015, 
    'The Lord of the Rings: The Fellowship of the Ring': 2001, 
    'Coco': 2017,
    'Goodfellas': 1990, 
    'The Good, the Bad and the Ugly': 1966, 
    'Toy Story 3': 2010,
    'Rear Window': 1954, 
    'Forrest Gump': 1994, 
    'Toy Story 4': 2019, 
    'City Lights': 1931,
    'Fight Club': 1999, 
    'Finding Nemo': 2003, 
    'Alien': 1979, 
    'The Lord of the Rings: The Two Towers': 2002,
    'How to Train Your Dragon': 2010, 
    'The Silence of the Lambs': 1991, 
    'Inception': 2010,
    'Up': 2009, 
    'The Shining': 1980, 
    'Star Trek': 2009, 
    'Seven': 1995, 
    'The Matrix': 1999, 
    'The Social Network': 2010,
    'The Apartment': 1960, 
    'Eighth Grade': 2018, 
    'Casablanca': 1942,
    "One Flew Over the Cuckoo's Nest": 1975, 
    'Argo': 2012, 
    'Seven Samurai': 1954,
    'Portrait of a Lady on Fire': 2019, 
    'Hell or High Water': 2016,
    "It's a Wonderful Life": 1946, 
    'Memento': 2000, 
    'Psycho': 1960,
    'Spider-Man: No Way Home': 2021, 
    "Singin' in the Rain": 1952, 
    'City of God': 2002,
    'The Farewell': 2019, 
    'Apocalypse Now': 1979, 
    'Saving Private Ryan': 1998,
    'Whiplash': 2014, 
    'Vertigo': 1958, 
    'Life Is Beautiful': 1997, 
    'Your Name': 2016,
    'Citizen Kane': 1941, 
    'The Green Mile': 1999, 
    'The Pianist': 2002, 
    'Taxi Driver': 1976,
    'Star Wars: Episode IV - A New Hope': 1977, 
    'Harry Potter and the Deathly Hallows Part 2': 2011,
    '2001: A Space Odyssey': 1968, 
    'Interstellar': 2014,
    'Spider-Man: Across the Spider-Verse': 2023, 
    'Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb': 1964,
    'Terminator 2: Judgment Day': 1991, 
    'Get Out': 2017, 
    'Double Indemnity': 1944,
    'Back to the Future': 1985, 
    'Selma': 2014, 
    'Raging Bull': 1980, 
    'Moonlight': 2016,
    'Chinatown': 1974, 
    'Lady Bird': 2017, 
    'Lawrence of Arabia': 1962, 
    'Dunkirk': 2017,
    'Paddington 2': 2017, 
    'Jaws': 1975,
    'The Professional': 1994, 
    'Black Panther': 2018, 
    'Some Like It Hot': 1959,
    'The Lion King': 1994, 'In the Mood for Love': 2000, 'Paths of Glory': 1957,
    'Gladiator': 2000, 'Arrival': 2016, 'American History X': 1998,
    'Inside Llewyn Davis': 2013, 'Toy Story': 1995, 'The Departed': 2006,
    'Marriage Story': 2019, 'The Prestige': 2006, 'Anatomy of a Fall': 2023,
    'The Third Man': 1949, 'Boyhood': 2014, 'The Wizard of Oz': 1939,
    'The Usual Suspects': 1995, 'Gravity': 2013, 'Roma': 2018,
    'Grave of the Fireflies': 1988, 'The Holdovers': 2023, 'Rashomon': 1950,
    'Everything Everywhere All at Once': 2022, 'North by Northwest': 1959,
    'The Intouchables': 2011, 'Minari': 2020, 'Blade Runner': 1982, 'Modern Times': 1936,
    'Past Lives': 2023, 'Blue Velvet': 1986, 'Once Upon a Time in the West': 1968,
    'Carol': 2015, '8-12': 1963, 'If Beale Street Could Talk': 2018,
    'On the Waterfront': 1954, 'Cinema Paradiso': 1988, 'First Cow': 2019,
    'Amadeus': 1984, 'Call Me by Your Name': 2017, 'Do the Right Thing': 1989,
    'Widows': 2018, 'Ran': 1985, '12 Years a Slave': 2013, 'Metropolis': 1927,
    'The Wolf of Wall Street': 2013, 'Unforgiven': 1992, 'Shoplifters': 2018,
    'Django Unchained': 2012, 'Brooklyn': 2015, 'Touch of Evil': 1958,
    "Won't You Be My Neighbor?": 2018, 'The Lives of Others': 2006, 'The Favourite': 2018,
    'Stalker': 1979, 'Sunset Boulevard': 1950, 'A Separation': 2011, 'Annie Hall': 1977,
    'Little Women (2019)': 2019, 'Phantom Thread': 2017,
    'Great Expectations': 1946, 'The Great Dictator': 1940, 'Tar': 2022,
    'The General': 1998, 'Witness for the Prosecution': 1957, 'Before Sunset': 2004,
    'Butch Cassidy and the Sundance Kid': 1969, 'Avengers: Infinity War': 2018,
    'The Shape of Water': 2017, 'To Kill a Mockingbird': 1962, 'Aliens': 1986,
    'The Handmaiden': 2016, 
    'Stagecoach': 1939, 
    'American Beauty': 1999,
    'Manchester by the Sea': 2016, 
    'All About Eve': 1950,
    'Once Upon a Time in Hollywood': 2019,  
    'Burning': 2018,
    'Fargo': 1996, 'Oldboy': 2003, 'Under the Skin': 2013, 'The 400 Blows': 1959,
    'Princess Mononoke': 1997, 'Her': 2013, 'The Dark Knight Rises': 2012,
    'No Country for Old Men': 2007, 'Ben-Hur': 1959, 'The Grand Budapest Hotel': 2014,
    'Bonnie and Clyde': 1967, 'La La Land': 2016, 'Beauty and the Beast': 1946,
    'Braveheart': 1995, 'Birdman or (The Unexpected Virtue of Ignorance)': 2014, 'Breathless': 1960, 'Joker': 2019,
    'Spotlight': 2015, 'Bicycle Thieves': 1948, 'Ex Machina': 2014,
    'O Brother, Where Art Thou?': 2000, 'Notorious': 1946, 'Das Boot': 1981,
    'Mulholland Dr': 2001, 'The Searchers': 1956, 'Inglourious Basterds': 2009,
    "Pan's Labyrinth": 2006, 'Hamilton': 2020, 'Amelie': 2001, 'Come and See': 1985,
    '3 Idiots': 2009, 'Million Dollar Baby': 2004, 'The Bridge on the River Kwai': 1957,
    'Brokeback Mountain': 2005, 'A Clockwork Orange': 1971, 'The Sting': 1973,
    'The Master': 2012, 'Wild Strawberries': 1957, 'Good Will Hunting': 1997,
    'Blade Runner 2049': 2017, 
    'High Noon': 1952, 
    'Requiem for a Dream': 2000,
    'The Deer Hunter': 1978, 
    'Inside Out 2': 2024,
    'Eternal Sunshine of the Spotless Mind': 2004, 
    'Avengers: Endgame': 2019,
    'Manhattan': 1979, 
    'Wall-E': 2008, 
    'The Great Escape': 1963, 
    'John Wick': 2014,
    'LA Confidential': 1997, 
    'Full Metal Jacket': 1987, 
    'The Zone of Interest': 2023,
    'Aguirre, the Wrath of God': 1972, 
    'Decision to Leave': 2022,
    'Strangers on a Train': 1951, 
    'The Maltese Falcon': 1941, 
    'Poor Things': 2023,
    'Heat': 1995, 
    'Reservoir Dogs': 1992, 
    'Kill Bill: Vol 1': 2003, 
    'Barry Lyndon': 1975,
    'The Iron Claw': 2023, 
    'Platoon': 1986, 
    'Godzilla Minus One': 2023, 
    'Seven': 1995,
    'Hunt for the Wilderpeople': 2016, 
    'Bottoms': 2023, 
    'The Wild Bunch': 1969,
    '1917': 2019, 
    'Oppenheimer': 2023, 
    'Rebecca': 1940, 
    'Talk to Me': 2022,
    'The Treasure of the Sierra Madre': 1948, 
    'The Wrestler': 2008, 
    'Dog Day Afternoon': 1975
}

# Define Functions

In [36]:
def _create_driver() -> webdriver.Chrome:
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1920,1080")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
    options.page_load_strategy = "eager"
    driver = webdriver.Chrome(options=options)
    driver.set_page_load_timeout(20)
    return driver
 
def get_movie_review_url(movie_name: str, year: int, driver: webdriver.Chrome) -> str:
    """
    Normalizes the movie name, constructs the Metacritic critic reviews URL
    sorted by Recently Added, validates the page exists, and returns the URL.
    Tries the URL without year first, then falls back to appending the year to the slug.
    """
    normalized = movie_name.lower()
    normalized = normalized.replace(" ", "-")
    normalized = re.sub(r"[^a-z0-9-]", "", normalized)
    normalized = normalized.strip("-")                    
    
    def build_url(slug: str) -> str:
        params = urllib.parse.urlencode({
            "filter": "All Reviews",
            "sort-by": "Recently Added"
        })
        return f"https://www.metacritic.com/movie/{slug}/critic-reviews?{params}"

    def try_url(url: str) -> bool:
        driver.get(url)
        page_title = driver.title.lower()
        current_url = driver.current_url
        if "page not found" in page_title or "404" in page_title:
            return False
        if "critic-reviews" not in current_url:
            return False
        return True

    # First attempt: without year (e.g. /movie/alien/critic-reviews)
    url_without_year = build_url(normalized)
    if try_url(url_without_year):
        return url_without_year

    # Second attempt: with year appended to slug (e.g. /movie/alien-1979/critic-reviews)
    url_with_year = build_url(f"{normalized}-{year}")
    if try_url(url_with_year):
        return url_with_year

    url_with_re_release = build_url(f"{normalized}-re-release")
    if try_url(url_with_re_release):
        return url_with_re_release
    
    raise ValueError(
        f"Movie '{movie_name}' not found on Metacritic. "
        f"Tried with and without year ({year}) and re-release."
    )    
 
def get_critic_reviews(
    movie_url: str,
    driver: webdriver.Chrome,
    movie_name: str = "movie",
) -> str:
    driver.get(movie_url)

    # Wait until the first batch of review cards loads
    try:
        WebDriverWait(driver, 5).until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, 'div[data-testid="review-card"]')
            )
        )
    except Exception:
        raise ConnectionError(
            "Timed out waiting for review cards to load. "
            "The page may be blocking automated access or the URL is incorrect."
        )

    # Scroll down incrementally to trigger lazy-loading of all reviews
    last_count = 0
    for _ in range(20):  # max 20 scroll attempts to avoid infinite loops
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(3)  # wait for new reviews to load after each scroll

        current_count = len(driver.find_elements(By.CSS_SELECTOR, 'div[data-testid="review-card"]'))

        if current_count >= 50:
            break  # we have enough reviews, stop scrolling
        if current_count == last_count:
            break  # no new reviews loaded, we've hit the bottom
        last_count = current_count

    soup = BeautifulSoup(driver.page_source, "html.parser")
    review_cards = soup.find_all("div", attrs={"data-testid": "review-card"})

    reviews = []
    counter = 1

    for card in review_cards:
        if counter > 50:
            break

        score_element = card.find(
            "div", attrs={"aria-label": re.compile(r"Metascore \d+ out of 100")}
        )
        if score_element is None:
            score_span = card.find("span", attrs={"data-v-033f7d05": True})
            score = score_span.get_text(strip=True) if score_span else None
        else:
            aria_label = score_element.get("aria-label", "")
            match = re.search(r"Metascore (\d+) out of 100", aria_label)
            score = match.group(1) if match else None

        quote_div = card.find("div", class_="review-card__quote")
        review_text = quote_div.get_text(strip=True) if quote_div else None

        if score is None or review_text is None:
            continue

        reviews.append({
            "review_id": counter,
            "user_rating": int(score),
            "review_text": review_text,
        })
        counter += 1

    filename = f"critic-reviews-metacritic/{movie_name}-critic-reviews.csv"
    with open(filename, "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=["review_id", "user_rating", "review_text"])
        writer.writeheader()
        writer.writerows(reviews)

    print(f"Saved {len(reviews)} reviews to '{filename}'")
    return filename

# Execute Functions

In [41]:
problematic_movies = []
driver = _create_driver()
 
try:
    for movie, year in islice(movies.items(), 0, None):
        csv_path = f"critic-reviews-metacritic/{movie}-critic-reviews.csv"
 
        # Checkpoint: skip if already scraped
        if os.path.exists(csv_path):
            print(f"[SKIP] '{movie}' already scraped — found {csv_path}")
            logging.info(f"SKIP | '{movie}' already scraped at {csv_path}")
            continue
 
        try:
            url = get_movie_review_url(movie, year, driver)
            get_critic_reviews(url, driver, movie_name=movie)
            logging.info(f"OK | '{movie}' saved to {csv_path}")
 
        except (ValueError, ConnectionError) as e:
            print(f"[ERROR] Skipping '{movie}': {e}")
            logging.error(f"ERROR | '{movie}' | {e}")
            problematic_movies.append({"movie": movie, "reason": str(e)})
            continue
 
        time.sleep(2)
 
finally:
    driver.quit()
 

[SKIP] 'The Shawshank Redemption' already scraped — found critic-reviews-metacritic/The Shawshank Redemption-critic-reviews.csv
[SKIP] 'Parasite' already scraped — found critic-reviews-metacritic/Parasite-critic-reviews.csv
[SKIP] 'Star Wars: Episode V - The Empire Strikes Back' already scraped — found critic-reviews-metacritic/Star Wars: Episode V - The Empire Strikes Back-critic-reviews.csv
[SKIP] 'The Godfather' already scraped — found critic-reviews-metacritic/The Godfather-critic-reviews.csv
[SKIP] 'Top Gun: Maverick' already scraped — found critic-reviews-metacritic/Top Gun: Maverick-critic-reviews.csv
[SKIP] 'The Godfather: Part II' already scraped — found critic-reviews-metacritic/The Godfather: Part II-critic-reviews.csv
[SKIP] 'Spider-Man: Into the Spider-Verse' already scraped — found critic-reviews-metacritic/Spider-Man: Into the Spider-Verse-critic-reviews.csv
[SKIP] 'Raiders of the Lost Ark' already scraped — found critic-reviews-metacritic/Raiders of the Lost Ark-critic-

In [42]:
# Summary

print("\n── Scrape Complete ──────────────────────────────────────────────────")
print(f"Total movies attempted : {len(movies)}")
print(f"Problematic movies     : {len(problematic_movies)}")
 
if problematic_movies:
    print("\nThe following movies had errors and were skipped:")
    for entry in problematic_movies:
        print(f"  • {entry['movie']}: {entry['reason']}")
    logging.warning(
        f"Scrape finished with {len(problematic_movies)} problematic movies: "
        + ", ".join(e["movie"] for e in problematic_movies)
    )
else:
    print("All movies scraped successfully!")
    logging.info("Scrape finished — no problematic movies.")


── Scrape Complete ──────────────────────────────────────────────────
Total movies attempted : 239
Problematic movies     : 5

The following movies had errors and were skipped:
  • 8-12: Movie '8-12' not found on Metacritic. Tried with and without year (1963) and re-release.
  • The Great Dictator: Movie 'The Great Dictator' not found on Metacritic. Tried with and without year (1940) and re-release.
  • The 400 Blows: Movie 'The 400 Blows' not found on Metacritic. Tried with and without year (1959) and re-release.
  • Bicycle Thieves: Movie 'Bicycle Thieves' not found on Metacritic. Tried with and without year (1948) and re-release.
  • Aguirre, the Wrath of God: Movie 'Aguirre, the Wrath of God' not found on Metacritic. Tried with and without year (1972) and re-release.
